Script for source space setup, forward/inverse modeling, and source estimation (all in steps).

**Summary:** This script iterates over MEG_manifest.csv (one row per MEG run), and for each subject × session it:
- Validates required upstream products exist (FreeSurfer recon, BEM solution, session trans.fif).
- Loads the MEG raw FIF + BEM solution + trans (head→MRI).
- Builds a mixed source space (optional cortical surface + volumetric grid).
- (Manually) transforms the source space coordinates from MRI→HEAD and saves both versions.
- Builds a forward model, computes a noise covariance from fixed-length epochs, builds an inverse operator, applies it to the full recording to get an STC.
- Saves: the raw vector STC, a devectorized magnitude STC, and a time-averaged NIfTI volume.

[Runtime: approx. 2-2.5 min per MEG scan file.]

-------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, re, subprocess
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mne
from mne import read_trans
from mne import read_source_spaces
from mne.io.constants import FIFF
from mne.transforms import Transform, apply_trans, invert_transform
from mne import write_source_spaces
from mne import VolSourceEstimate
from mne.minimum_norm import make_inverse_operator, apply_inverse_raw, write_inverse_operator
import nibabel as nib
from nilearn.datasets import load_mni152_template
import nilearn.plotting as plotting
import csv
import warnings

In [ ]:
##### SET UP ENVIRONMENTAL VARIABLES FOR FREESURFER:
freesurfer_config = config.get("freesurfer", {})
FREESURFER_HOME = Path(freesurfer_config.get("home", "/opt/freesurfer-7.4.1")).expanduser()
FS_LICENSE = Path(freesurfer_config.get("license", FREESURFER_HOME / "license.txt")).expanduser()
SUBJECTS_DIR = Path(freesurfer_config.get("subjects_dir", FREESURFER_HOME / "subjects")).expanduser()
CHECK_FS_VERSION = bool(freesurfer_config.get("check_version", True))
if not FREESURFER_HOME.exists():
    raise FileNotFoundError(f"FREESURFER_HOME not found: {FREESURFER_HOME}")
if not FS_LICENSE.exists():
    raise FileNotFoundError(f"FreeSurfer license file not found: {FS_LICENSE}")
if not SUBJECTS_DIR.exists():
    raise FileNotFoundError(f"FreeSurfer SUBJECTS_DIR not found: {SUBJECTS_DIR}")
os.environ["FREESURFER_HOME"] = str(FREESURFER_HOME)
os.environ["FS_LICENSE"] = str(FS_LICENSE)
os.environ["SUBJECTS_DIR"] = str(SUBJECTS_DIR)
fs_bin_dir = FREESURFER_HOME / "bin"
os.environ["PATH"] = f"{fs_bin_dir}:{os.environ.get('PATH', '')}"

subprocess.run("recon-all --version", shell=True, check=True)

In [ ]:
# --------------------------------------------------------------------
### SET PARAMETERS:

HARD_STOP = config['hard_errors']
RANDOM_SEED = config['random_seed']

SUBSET = config['subset']

OVERWRITE = config['overwrite_source_estimation']


# SOURCE SPACING PARAMETERS:
INCLUDE_CORTICAL_SURFACE = config['source_spacing']['include_cortical_surface']
TARGET_SURFACE_FILE = config['source_spacing']['target_surface_file']
CORTICAL_RESOLUTION = config['source_spacing']['cortical_resolution']
VOLUMETRIC_RESOLUTION = config['source_spacing']['volumetric_resolution']

# EPOCHING PARAMETERS:
NOISE_EPOCH_LENGTH  = config['noise_epoching']['noise_epoch_length']
NOISE_EPOCH_OVERLAP = config['noise_epoching']['noise_epoch_overlap']

# INVERSE MODELING PARAMETERS:
INVERSE_MODEL_METHOD = config['inverse_modeling']['inverse_model_method']
INVERSE_MODEL_ORIENTATION = config['inverse_modeling']['inverse_model_orientation']
INVERSE_MODEL_DEPTH = config['inverse_modeling']['inverse_model_depth']

ASSUMED_SNR = config['source_estimation']['assumed_SNR']


# OUTPUT PROCESSING PARAMETERS:
DEVECTORIZE = config['source_estimation']['devectorize']
DOWNSAMPLING_RATE = config['source_estimation']['downsampling_rate']
SAVE_RAW_DATA = config['source_estimation']['save_raw_data']
SAVE_RAW_SOURCE_ESTIMATE = config['source_estimation']['write_full_source_estimate']
SAVE_FLATTENED_DATA = config['source_estimation']['save_flattened_copy']


# --------------------------------------------------------------------
### SET PATHS:

ROOT_DIR = Path(config['root_output_directory'])

### INPUTS:
RUN_MANIFEST_PATH = Path(ROOT_DIR) / 'subject_manifest.csv'
MEG_DATA_DIR = config['MEG_data_directory']
MEG_PARAMETERS_PATH = Path(ROOT_DIR) / 'MEG_manifest.csv'

COREGISTRATION_DATA_DIRECTORY = Path(ROOT_DIR) / config['coreg_output_dir']

### OUTPUTS:
# Source estimation stage outputs are split into separate top-level folders under ROOT_DIR:
SOURCE_SPACES_DIR = ROOT_DIR / config['source_space_output_dir']
FORWARD_MODELS_DIR = ROOT_DIR / config['forward_modeling_output_dir']
INVERSE_MODELS_DIR = ROOT_DIR / config['inverse_modeling_output_dir']
SOURCE_ESTIMATES_DIR = ROOT_DIR / config['source_estimate_output_dir']
for directory in [SOURCE_SPACES_DIR, FORWARD_MODELS_DIR, INVERSE_MODELS_DIR, SOURCE_ESTIMATES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

QC_OUTPUT_DIR = ROOT_DIR / config['SRC_QC_output_subdir']

# --------------------------------------------------------------------
### INITIALIZATION:

# Load DataFrames:
RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
MEG_runs = pd.read_csv(MEG_PARAMETERS_PATH)

In [ ]:
### DIAGNOSTIC SUBSETTING (if enabled):
if type(SUBSET) == int and SUBSET > 0:
    print(f"\nDIAGNOSTIC SUBSETTING ENABLED: Running only {SUBSET} test subjects/files:")
    MEG_runs = MEG_runs.head(SUBSET).copy()
    display(MEG_runs)

---------
Retrieve filepaths to input files:

In [ ]:
# --------------------------------------------------------------------
### RESOLVE MEG FILE PATHS (recursive search under MEG_DATA_DIR)
# --------------------------------------------------------------------

if "MEG_fullpath" in MEG_runs.columns:
    print("[info] MEG_fullpath column already present in MEG_runs; skipping MEG_DATA_DIR scan.")
else:
    print(f"[scan] Resolving MEG .fif paths under {MEG_DATA_DIR}...")

    # Initialize column
    MEG_runs['MEG_fullpath'] = np.nan

    all_meg_fif_paths = []

    for root, dirnames, filenames in os.walk(MEG_DATA_DIR):
        for filename in filenames:
            if filename.endswith('.fif'):
                all_meg_fif_paths.append(Path(root) / filename)

    print(f"[scan] Found {len(all_meg_fif_paths)} .fif files under {MEG_DATA_DIR}")

    # Build an index from base name (without '.fif') to path
    MEG_file_index = {}
    duplicate_basenames = []

    for path in all_meg_fif_paths:
        base_name = path.name[:-4]  # strip ".fif"
        if base_name in MEG_file_index and MEG_file_index[base_name] != path:
            duplicate_basenames.append(base_name)
        else:
            MEG_file_index[base_name] = path

    if duplicate_basenames:
        print("[warn] Multiple .fif files found for some base names:")
        for base_name in sorted(set(duplicate_basenames)):
            print(f"   - {base_name}")
        if HARD_STOP:
            raise RuntimeError(
                "Duplicate MEG base names found under MEG_DATA_DIR; "
                "please resolve these conflicts before continuing.")

    # Map each MEG_runs row to a concrete path
    missing_meg_entries = []

    for dataframe_index, run_row in MEG_runs.iterrows():
        meg_filename_base = run_row['MEG_filename']  # BIDS-style base, no extension
        if meg_filename_base in MEG_file_index:
            MEG_runs.at[dataframe_index, 'MEG_fullpath'] = str(MEG_file_index[meg_filename_base])
        else:
            missing_meg_entries.append((dataframe_index, run_row['subject_ID'], meg_filename_base))

    if missing_meg_entries:
        print("[warn] Missing MEG files for the following runs (row_index, subject_ID, MEG_filename):")
        for dataframe_index, subject_ID, meg_filename_base in missing_meg_entries[:10]:
            print(f"   - {dataframe_index}: {subject_ID} | {meg_filename_base}")
        print(f"[warn] Total missing MEG runs: {len(missing_meg_entries)}")

        if HARD_STOP:
            raise FileNotFoundError(
                f"Missing MEG .fif files for {len(missing_meg_entries)} run(s); "
                "see warnings above.")
        else:
            # Drop missing rows from MEG_runs when HARD_STOP is False
            missing_indices = [idx for idx, _, _ in missing_meg_entries]
            MEG_runs = MEG_runs.drop(index=missing_indices).reset_index(drop=True)
            print(f"[info] Dropped {len(missing_indices)} run(s) with missing MEG files; "
                  f"{len(MEG_runs)} run(s) remain.")

------------
Main code:

In [ ]:
# --------------------------------------------------------------------
# SOURCE SPACE, FORWARD MODEL, INVERSE OPERATOR, SOURCE ESTIMATION + QC
#
# QC outputs:
#   - QC_OUTPUT_DIR/QC_coreg_metrics.csv  (one row per subject_ID x session_ID)
#   - <subject_ID>_<session_ID>_QC_coreg_fid_hpi_slice_overlay.png
#
# IMPORTANT:
#   - MNE coordinates are in METERS
#   - FreeSurfer/NiBabel affines are in MILLIMETERS
#   - We convert MRI(surface RAS) points from meters -> mm before mapping to voxels.
# --------------------------------------------------------------------

OVERWRITE = config.get("overwrite_source_estimation", False)

SOURCE_ESTIMATION_ISSUES = []

QC_OUTPUT_DIR = Path(QC_OUTPUT_DIR)
QC_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QC_CSV_PATH = QC_OUTPUT_DIR / "QC_coreg_metrics.csv"

_cf_name = {
    FIFF.FIFFV_COORD_UNKNOWN: "unknown",
    FIFF.FIFFV_COORD_DEVICE: "MEG device",
    FIFF.FIFFV_COORD_HEAD: "head",
    FIFF.FIFFV_COORD_MRI: "MRI (surface RAS)"}

def _as_mm(x_m):
    return float(x_m) * 1000.0

def _get_head_to_mri(trans):
    # Return HEAD->MRI (surface RAS) 4x4 in MNE convention (units: meters)
    if trans["from"] == FIFF.FIFFV_COORD_HEAD and trans["to"] == FIFF.FIFFV_COORD_MRI:
        return trans["trans"]
    if trans["from"] == FIFF.FIFFV_COORD_MRI and trans["to"] == FIFF.FIFFV_COORD_HEAD:
        return invert_transform(trans)["trans"]
    raise RuntimeError(f"Unexpected trans frames: from={trans['from']} to={trans['to']}")

def _get_device_to_head(info):
    dht = info.get("dev_head_t", None)
    if dht is None:
        return None
    return dht["trans"]

def _pairwise_dist_mm(pts_m):
    pts_m = np.asarray(pts_m, float)
    if pts_m.ndim != 2 or pts_m.shape[0] < 2:
        return np.array([], float)
    out = []
    for i in range(pts_m.shape[0]):
        for j in range(i + 1, pts_m.shape[0]):
            out.append(_as_mm(np.linalg.norm(pts_m[i] - pts_m[j])))
    return np.asarray(out, float)

def _collect_fid_hpi_as_head(info):
    """
    Returns:
      fid_head: dict { 'LPA'/'NAS'/'RPA': (3,) meters in HEAD }
      hpi_head: (N,3) meters in HEAD
      inv: dict diagnostics
    """
    dig = info.get("dig", None)
    if not dig:
        return {}, np.empty((0, 3), float), {
            "dig_present": False,
            "dig_n_total": 0,
            "dev_head_t_present": bool(info.get("dev_head_t", None) is not None),
            "dig_kind_counts": {}}

    dev_to_head = _get_device_to_head(info)

    fid_head = {}
    hpi_head = []
    kind_counts = {}

    ident_to_label = {
        FIFF.FIFFV_POINT_LPA: "LPA",
        FIFF.FIFFV_POINT_NASION: "NAS",
        FIFF.FIFFV_POINT_RPA: "RPA"}

    for d in dig:
        kind = int(d["kind"])
        coord_frame = int(d["coord_frame"])
        r = np.asarray(d["r"], float)

        kind_counts[kind] = kind_counts.get(kind, 0) + 1

        # Convert to HEAD where possible
        if coord_frame == FIFF.FIFFV_COORD_HEAD:
            r_head = r
        elif coord_frame == FIFF.FIFFV_COORD_DEVICE and dev_to_head is not None:
            r_head = apply_trans(dev_to_head, r.reshape(1, 3)).ravel()
        else:
            continue

        if kind == FIFF.FIFFV_POINT_CARDINAL:
            ident = int(d.get("ident", -1))
            lab = ident_to_label.get(ident, None)
            if lab is not None:
                fid_head[lab] = r_head

        elif kind == FIFF.FIFFV_POINT_HPI:
            hpi_head.append(r_head)

    hpi_head = np.asarray(hpi_head, float) if len(hpi_head) else np.empty((0, 3), float)

    inv = {
        "dig_present": True,
        "dig_n_total": int(len(dig)),
        "dev_head_t_present": bool(info.get("dev_head_t", None) is not None),
        "dig_kind_counts": dict(kind_counts)}
    return fid_head, hpi_head, inv

for subject_ID, subject_meg_runs in MEG_runs.groupby("subject_ID"):

    print("\n" + "_" * 100, flush=True)
    print(f"=== Processing subject: {subject_ID} ===", flush=True)

    fs_subject_dir = SUBJECTS_DIR / subject_ID
    if not fs_subject_dir.exists():
        message = f"FreeSurfer directory missing: {fs_subject_dir}"
        print("ERROR:", message, flush=True)
        if HARD_STOP:
            raise FileNotFoundError(message)
        SOURCE_ESTIMATION_ISSUES.append(subject_ID)
        continue

    coreg_subject_dir = COREGISTRATION_DATA_DIRECTORY / subject_ID
    if not coreg_subject_dir.exists():
        message = f"Coregistration directory missing: {coreg_subject_dir}"
        print("ERROR:", message, flush=True)
        if HARD_STOP:
            raise FileNotFoundError(message)
        SOURCE_ESTIMATION_ISSUES.append(subject_ID)
        continue

    subject_source_space_dir = SOURCE_SPACES_DIR / subject_ID
    subject_forward_dir = FORWARD_MODELS_DIR / subject_ID
    subject_inverse_dir = INVERSE_MODELS_DIR / subject_ID
    subject_estimates_dir = SOURCE_ESTIMATES_DIR / subject_ID

    for directory in [subject_source_space_dir, subject_forward_dir, subject_inverse_dir, subject_estimates_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    MRI_data_path = fs_subject_dir / "mri" / "T1.mgz"
    if not MRI_data_path.exists():
        message = f"MRI file not found for {subject_ID}: {MRI_data_path}"
        print("ERROR:", message, flush=True)
        if HARD_STOP:
            raise FileNotFoundError(message)
        SOURCE_ESTIMATION_ISSUES.append(subject_ID)
        continue

    BEM_solution_path = coreg_subject_dir / f"{subject_ID}_bem-sol.fif"
    if not BEM_solution_path.exists():
        message = f"BEM solution not found for {subject_ID}: {BEM_solution_path}"
        print("ERROR:", message, flush=True)
        if HARD_STOP:
            raise FileNotFoundError(message)
        SOURCE_ESTIMATION_ISSUES.append(subject_ID)
        continue

    subject_meg_runs = subject_meg_runs[subject_meg_runs["MEG_fullpath"].notna()]
    if subject_meg_runs.empty:
        print(f"  - No MEG runs with valid paths for {subject_ID}. Skipping subject.", flush=True)
        SOURCE_ESTIMATION_ISSUES.append(subject_ID)
        continue

    for _, row in subject_meg_runs.iterrows():

        session_ID = row["MEG_session_ID"]
        MEG_data_path = Path(row["MEG_fullpath"])
        prefix = f"{subject_ID}_{session_ID}"

        print("\n" + "_" * 98, flush=True)
        print(f"--- Subject {subject_ID} | Session {session_ID} ---", flush=True)
        print(f"  - MEG input file: {MEG_data_path}", flush=True)

        if "trans_fullpath" in MEG_runs.columns and pd.notna(row.get("trans_fullpath", np.nan)):
            TRANS_file_path = Path(row["trans_fullpath"])
        else:
            TRANS_file_path = coreg_subject_dir / f"{subject_ID}_{session_ID}_trans.fif"

        if not TRANS_file_path.exists():
            message = f"TRANS file not found for {subject_ID}, session {session_ID}: {TRANS_file_path}"
            print("ERROR:", message, flush=True)
            if HARD_STOP:
                raise FileNotFoundError(message)
            SOURCE_ESTIMATION_ISSUES.append(subject_ID)
            continue

        input_MEG = mne.io.read_raw_fif(str(MEG_data_path), preload=True)
        BEM_solution = mne.read_bem_solution(str(BEM_solution_path))
        trans_file = read_trans(str(TRANS_file_path))

        print("************************************************************************************************************", flush=True)
        print("[frame-audit] TRANS file:", flush=True)
        print(f"  - trans['from'] = {_cf_name.get(trans_file['from'], trans_file['from'])} ({trans_file['from']})", flush=True)
        print(f"  - trans['to']   = {_cf_name.get(trans_file['to'], trans_file['to'])} ({trans_file['to']})", flush=True)
        print("************************************************************************************************************", flush=True)

        trans_frames = {trans_file["from"], trans_file["to"]}
        if not ({FIFF.FIFFV_COORD_HEAD, FIFF.FIFFV_COORD_MRI} <= trans_frames):
            raise RuntimeError("TRANS sanity-check failed: expected a HEAD<->MRI transform.")

        if not (isinstance(ASSUMED_SNR, (int, float)) and float(ASSUMED_SNR) > 0):
            raise ValueError(f"ASSUMED_SNR must be a positive number; got {ASSUMED_SNR!r}.")

        # -------------------------
        # QC metrics + plot
        # -------------------------
        qc_metrics = {
            "subject_ID": subject_ID,
            "session_ID": session_ID,
            "MEG_fullpath": str(MEG_data_path),
            "TRANS_fullpath": str(TRANS_file_path),
            "ASSUMED_SNR": float(ASSUMED_SNR),
            "trans_from": int(trans_file["from"]),
            "trans_to": int(trans_file["to"])}

        fid_head, hpi_head, dig_inv = _collect_fid_hpi_as_head(input_MEG.info)

        qc_metrics["dig_present"] = bool(dig_inv.get("dig_present", False))
        qc_metrics["dig_n_total"] = int(dig_inv.get("dig_n_total", 0))
        qc_metrics["dev_head_t_present"] = bool(dig_inv.get("dev_head_t_present", False))
        qc_metrics["n_fiducials_found"] = int(len(fid_head))
        qc_metrics["n_hpi_found"] = int(hpi_head.shape[0])
        qc_metrics["dig_n_kind_cardinal"] = int(dig_inv.get("dig_kind_counts", {}).get(int(FIFF.FIFFV_POINT_CARDINAL), 0))
        qc_metrics["dig_n_kind_hpi"] = int(dig_inv.get("dig_kind_counts", {}).get(int(FIFF.FIFFV_POINT_HPI), 0))

        try:
            if all(k in fid_head for k in ("LPA", "RPA", "NAS")):
                LPA = fid_head["LPA"]
                RPA = fid_head["RPA"]
                NAS = fid_head["NAS"]
                qc_metrics["fid_LPA_RPA_mm"] = _as_mm(np.linalg.norm(LPA - RPA))
                qc_metrics["fid_NAS_LPA_mm"] = _as_mm(np.linalg.norm(NAS - LPA))
                qc_metrics["fid_NAS_RPA_mm"] = _as_mm(np.linalg.norm(NAS - RPA))
            else:
                qc_metrics["fid_distance_error"] = "Missing one or more of LPA/NAS/RPA in info['dig']."

            hpi_pw = _pairwise_dist_mm(hpi_head)
            if hpi_pw.size:
                qc_metrics["hpi_pw_min_mm"] = float(np.min(hpi_pw))
                qc_metrics["hpi_pw_median_mm"] = float(np.median(hpi_pw))
                qc_metrics["hpi_pw_max_mm"] = float(np.max(hpi_pw))
            else:
                qc_metrics["hpi_pw_error"] = "Not enough HPI points for pairwise distances."

            head_to_mri = _get_head_to_mri(trans_file)
            R = head_to_mri[:3, :3]
            t = head_to_mri[:3, 3]
            qc_metrics["trans_rot_det"] = float(np.linalg.det(R))
            qc_metrics["trans_trans_mag_mm"] = _as_mm(np.linalg.norm(t))
        except Exception as e:
            qc_metrics["numeric_qc_error"] = str(e)

        try:
            t1_img = nib.load(str(MRI_data_path))
            t1_data = t1_img.get_fdata()

            vox2ras_tkr = None
            if hasattr(t1_img, "header") and hasattr(t1_img.header, "get_vox2ras_tkr"):
                vox2ras_tkr = t1_img.header.get_vox2ras_tkr()

            if vox2ras_tkr is None:
                vox2ras = t1_img.affine
                inv_vox2ras = np.linalg.inv(vox2ras)
                qc_metrics["overlay_affine_used"] = "img.affine (fallback)"
            else:
                inv_vox2ras = np.linalg.inv(vox2ras_tkr)
                qc_metrics["overlay_affine_used"] = "vox2ras_tkr (FreeSurfer surface RAS)"

            head_to_mri = _get_head_to_mri(trans_file)

            pts_head = []
            labels = []

            for lab in ("LPA", "NAS", "RPA"):
                if lab in fid_head:
                    pts_head.append(fid_head[lab])
                    labels.append(lab)

            for i in range(hpi_head.shape[0]):
                pts_head.append(hpi_head[i])
                labels.append(f"HPI{i+1}")

            if len(pts_head) == 0:
                raise RuntimeError("No fiducials/HPI points available to plot.")

            pts_head = np.asarray(pts_head, float)

            pts_mri_m = apply_trans(head_to_mri, pts_head)
            pts_mri_mm = pts_mri_m * 1000.0

            vox = nib.affines.apply_affine(inv_vox2ras, pts_mri_mm)

            x = np.clip(vox[:, 0], 0, t1_data.shape[0] - 1)
            y = np.clip(vox[:, 1], 0, t1_data.shape[1] - 1)
            z = np.clip(vox[:, 2], 0, t1_data.shape[2] - 1)

            sag_x = int(np.median(x))
            cor_y = int(np.median(y))
            axi_z = int(np.median(z))

            fig = plt.figure(figsize=(14, 4.5))

            ax1 = fig.add_subplot(1, 3, 1)
            ax1.imshow(t1_data[sag_x, :, :].T, cmap="gray", origin="lower")
            ax1.scatter(y, z, s=80, alpha=0.9)
            for lab, yy, zz in zip(labels, y, z):
                ax1.text(yy + 2, zz + 2, lab, fontsize=10,
                         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1.5))
            ax1.set_title(f"sagittal x={sag_x}")
            ax1.set_axis_off()

            ax2 = fig.add_subplot(1, 3, 2)
            ax2.imshow(t1_data[:, cor_y, :].T, cmap="gray", origin="lower")
            ax2.scatter(x, z, s=80, alpha=0.9)
            for lab, xx, zz in zip(labels, x, z):
                ax2.text(xx + 2, zz + 2, lab, fontsize=10,
                         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1.5))
            ax2.set_title(f"coronal y={cor_y}")
            ax2.set_axis_off()

            ax3 = fig.add_subplot(1, 3, 3)
            ax3.imshow(t1_data[:, :, axi_z].T, cmap="gray", origin="lower")
            ax3.scatter(x, y, s=80, alpha=0.9)
            for lab, xx, yy in zip(labels, x, y):
                ax3.text(xx + 2, yy + 2, lab, fontsize=10,
                         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1.5))
            ax3.set_title(f"axial z={axi_z}")
            ax3.set_axis_off()

            plt.suptitle(f"{prefix} coreg QC: fiducials + HPI (via trans)")
            plt.tight_layout(rect=[0, 0.02, 1, 0.90])

            overlay_path = QC_OUTPUT_DIR / f"{prefix}_QC_coreg_fid_hpi_slice_overlay.png"
            fig.savefig(overlay_path, dpi=150)
            plt.close(fig)

        except Exception as e:
            qc_metrics["overlay_error"] = str(e)
            print(f"[QC][warn] Overlay not generated for {prefix}: {e}", flush=True)

        try:
            row_df = pd.DataFrame([qc_metrics])
            if QC_CSV_PATH.exists():
                row_df.to_csv(QC_CSV_PATH, mode="a", header=False, index=False)
            else:
                row_df.to_csv(QC_CSV_PATH, mode="w", header=True, index=False)
            print(f"[QC] Appended QC row -> {QC_CSV_PATH}", flush=True)
        except Exception as e:
            print(f"[QC][warn] Could not append QC row for {prefix}: {e}", flush=True)

        # -------------------------
        # Source space / forward / inverse / STC
        # -------------------------
        if INCLUDE_CORTICAL_SURFACE:
            cortical_source_space = mne.setup_source_space(
                subjects_dir=str(SUBJECTS_DIR),
                subject=subject_ID,
                surface=TARGET_SURFACE_FILE,
                spacing=CORTICAL_RESOLUTION,
                n_jobs=-1)
            cortical_dicts = cortical_source_space.copy()
        else:
            cortical_dicts = []

        volumetric_source_space = mne.setup_volume_source_space(
            subjects_dir=str(SUBJECTS_DIR),
            subject=subject_ID,
            mri=str(MRI_data_path),
            bem=str(BEM_solution_path),
            surface=None,
            pos=VOLUMETRIC_RESOLUTION)
        volumetric_dicts = volumetric_source_space.copy()

        final_source_space = mne.source_space.SourceSpaces(cortical_dicts + volumetric_dicts)

        forward_model = mne.make_forward_solution(
            input_MEG.info,
            bem=BEM_solution,
            trans=trans_file,
            src=final_source_space,
            eeg=False,
            meg=True)

        downsampled_MEG = input_MEG.copy()
        downsampled_MEG.resample(DOWNSAMPLING_RATE, npad="auto")

        epochs = mne.make_fixed_length_epochs(
            downsampled_MEG,
            duration=NOISE_EPOCH_LENGTH,
            overlap=NOISE_EPOCH_OVERLAP)
        epochs.apply_baseline((None, None))
        noise_cov_matrix = mne.compute_covariance(epochs)

        depth_value = INVERSE_MODEL_DEPTH
        if isinstance(depth_value, str):
            dv = depth_value.strip().lower()
            if dv in ("auto",):
                depth_mode = "auto"
                depth_param = None
            elif dv in ("none", "no_depth", "no-depth", "null", ""):
                depth_mode = "none"
                depth_param = None
            else:
                depth_param = float(depth_value)
                depth_mode = "numeric"
        elif isinstance(depth_value, (int, float)):
            depth_mode = "numeric"
            depth_param = float(depth_value)
        elif isinstance(depth_value, dict):
            depth_mode = "dict"
            depth_param = depth_value
        else:
            depth_mode = "auto"
            depth_param = None

        if depth_mode == "auto":
            inverse_operator = make_inverse_operator(downsampled_MEG.info, forward_model, noise_cov_matrix)
        elif depth_mode == "none":
            inverse_operator = make_inverse_operator(downsampled_MEG.info, forward_model, noise_cov_matrix, depth=None)
        else:
            inverse_operator = make_inverse_operator(downsampled_MEG.info, forward_model, noise_cov_matrix, depth=depth_param)

        lambda2 = 1.0 / (float(ASSUMED_SNR) ** 2)
        print(f"[inv] Using ASSUMED_SNR={ASSUMED_SNR} -> lambda2={lambda2:.6g}", flush=True)

        stc_RAW = apply_inverse_raw(
            downsampled_MEG,
            inverse_operator,
            lambda2=lambda2,
            method=INVERSE_MODEL_METHOD,
            pick_ori=INVERSE_MODEL_ORIENTATION)

        # ----------------------------------------------------------
        # 9) Optional: save full RAW vector-valued STC (pre de-vectorize)
        # ----------------------------------------------------------
        if SAVE_RAW_DATA:
            try:
                # Preserve your original naming convention here (underscore + RAW + .fif)
                source_estimate_raw_filepath = subject_estimates_dir / f"{subject_ID}_{session_ID}_source_estimate_RAW.fif"
                stc_RAW.save(str(source_estimate_raw_filepath), overwrite=OVERWRITE)
                print(f"\nSaved RAW (vector-valued) Source Estimate to: {source_estimate_raw_filepath}", flush=True)
            except Exception as e:
                print("\n!!! ERROR saving RAW Source Estimate (perhaps OVERWRITE=False and file exists).", flush=True)
                print("    Error:", e, flush=True)

        # ----------------------------------------------------------
        # 10) De-vectorize (magnitude across x,y,z)
        # ----------------------------------------------------------
        if DEVECTORIZE:
            stc_4d = stc_RAW.copy()
            stc_4d._data = np.linalg.norm(stc_RAW.data, axis=1)  # (n_sources, n_times)
            print("\nOriginal STC shape:\t", stc_RAW.data.shape, flush=True)
            print("De-vectorized shape:\t", stc_4d.data.shape, flush=True)
        else:
            stc_4d = stc_RAW.copy()
            print("\nDEVECTORIZE=False; using original vector-valued STC as stc_4d.", flush=True)

        # ----------------------------------------------------------
        # 11) SAVE the 4D de-vectorized STC (PRIMARY OUTPUT)
        # ----------------------------------------------------------
        if SAVE_RAW_SOURCE_ESTIMATE:
            try:
                source_estimate_4D_filepath = subject_estimates_dir / f"{subject_ID}-{session_ID}-Source_Estimate_4D_volume"
                stc_4d.save(str(source_estimate_4D_filepath), overwrite=OVERWRITE)
                print(f"\nSaved 4D Source Estimate to: {source_estimate_4D_filepath}", flush=True)
            except Exception as e:
                print("\n!!! ERROR saving 4D Source Estimate (perhaps OVERWRITE=False and file exists).", flush=True)
                print("    Error:", e, flush=True)

        # ----------------------------------------------------------
        # 12) Save flattened 3D STC as NIfTI (QC-only) (if toggled)
        # ----------------------------------------------------------
        if SAVE_FLATTENED_DATA:
            stc_3d = stc_4d.copy()
            stc_3d._data = stc_4d.data.mean(axis=1, keepdims=True)  # (n_sources, 1)

            try:
                source_estimate_3D_filepath = subject_estimates_dir / f"{subject_ID}-{session_ID}-Source_Estimate_3D_mean_over_time.nii"
                stc_3d.save_as_volume(
                    str(source_estimate_3D_filepath),
                    src=final_source_space,
                    mri_resolution=True,
                    overwrite=OVERWRITE)
                print(f"\n  --> Saved 3D QC volume to: {source_estimate_3D_filepath}", flush=True)
            except Exception as e:
                print("\n!!! ERROR saving 3D Source Estimate volume.", flush=True)
                print("    Error:", e, flush=True)

        if SOURCE_ESTIMATION_ISSUES:
            print("\n[summary] Subjects with source/forward/inverse issues:", flush=True)
            for s in sorted(set(SOURCE_ESTIMATION_ISSUES)):
                print("   -", s, flush=True)
        else:
            print("\n[summary] Source estimation stage completed without subject-level errors.", flush=True)

--------